# **前言**
- 数据源是来自和鲸社区的 3 份互不相关的电商数据集，所以分成 3 部分，每部分只对其中的一个数据集进行分析。

# **part 1**
- tmall_order_report.csv 这个数据集是订单数据，可供挖掘的纬度有订单时间、省份（收货地址），指标则有销售量、销售额、退款金额、退货率、成交率、地区分布、下单时间趋势等。

## **1. 数据理解与处理**

In [ ]:
# 导包，设置根路径
import pandas as pd
import os

from PIL.Image import item
from pyecharts import options as opts
from pyecharts.charts import Map, Bar, Line,Timeline,Pie
from pyecharts.components import Table
from pyecharts.options import ComponentTitleOpts
from pyecharts.faker import Faker

os.chdir(r'D:\python\PythonProject')

# 读取数据
data = pd.read_csv('./OrderFromTmall/data/tmall_order_report.csv')
data.head()

In [ ]:
data.info()

In [ ]:
# 列名有空格
data.columns = data.columns.str.strip()
data.columns

In [ ]:
# 无重复值
data[data.duplicated()].count()

In [ ]:
# 有空值：付款时间存在空值，表示订单未付款
data.isnull().sum()

In [ ]:
data['收货地址'] = data['收货地址'].str.replace('自治区|维吾尔|回族|壮族|省','',regex=True)       # 对省份进行清洗，方便可视化
data['收货地址'].unique()

## **2.数据可视化**

### **2.1 整体情况**

In [ ]:
result = {}
result['总订单数'] = data['订单编号'].count()
result['已完成订单数'] = data['订单编号'][data['订单付款时间'].notnull()].count()
result['未付款订单'] = data['订单编号'][data['订单付款时间'].isnull()].count()
result['退款订单'] = data['订单编号'][data['退款金额']>0].count()
result['总订单金额'] = data['总金额'][data['订单付款时间'].notnull()].sum()
result['总退款金额'] = data['退款金额'][data['订单付款时间'].notnull()].sum()
result['总实际收入金额'] = data['买家实际支付金额'][data['订单付款时间'].notnull()].sum()

In [ ]:
result

In [ ]:
table = Table()

headers = ['总订单数','总订单金额','总实际收入金额','已完成订单数','退款订单','总退款金额','成交率','退款率']
rows = [
    [
        result['总订单数'],f'{result['总订单金额']/10000:.2f} 万',f'{result['总实际收入金额']/10000:.2f} 万',result['已完成订单数'],result['退款订单'],f'{result['总退款金额']/10000:.2f} 万',f'{result['已完成订单数']/result['总订单数']:.2%}',f'{result['退款订单']/result['已完成订单数']:.2%}'
    ]
]
table.add(headers, rows)
table.set_global_opts(
    title_opts=ComponentTitleOpts(title='整体情况')
)
table.render_notebook()

### **2.2 地区分析**

In [ ]:
result2 = data[data['订单付款时间'].notnull()].groupby('收货地址').agg({'订单编号':'count'})
result21 = result2.to_dict()['订单编号']
c = (
    Map()
    .add('订单量',[*result21.items()],'china',is_map_symbol_show=False)
    .set_series_opts(label_opts=opts.LabelOpts(is_show=True))
    .set_global_opts(
        title_opts=opts.TitleOpts(title = '地区分布'),
        visualmap_opts=opts.VisualMapOpts(max_=1000)
    )
)

In [ ]:
c.render('./OrderFromTmall/可视化图/地区分布图.html')

### **2.3 时间分析**

In [ ]:
data['订单创建时间'] = pd.to_datetime(data['订单创建时间'])
data['订单付款时间'] = pd.to_datetime(data['订单付款时间'])

In [ ]:
result31 = data.groupby(data['订单创建时间'].apply(lambda x:x.strftime('%Y-%m-%d'))).agg({'订单编号':'count'}).to_dict()['订单编号']
c = (
    Line()
    .add_xaxis(list(result31.keys()))
    .add_yaxis('订单量',list(result31.values()))
    .set_series_opts(
        label_opts=opts.LabelOpts(is_show=False),
        markpoint_opts=opts.MarkPointOpts(
            data=[
                opts.MarkPointItem(type_='max',name='最大值')
            ]
        )
    )
    .set_global_opts(title_opts=opts.TitleOpts(title='每日订单量走势'))
)


In [ ]:
c.render('./OrderFromTmall/可视化图/每日订单量走势图.html')

- 从上图来看，2月份上半月由于受新冠疫情影响，订单量比较少，随着复工开展，下半月的订单量增长明显。

In [ ]:
result32 = data.groupby(data['订单创建时间'].apply(lambda x:x.strftime('%H'))).agg({'订单编号':'count'}).to_dict()['订单编号']
x = [*result32.keys()]
y = [*result32.values()]
c = (
    Bar()
    .add_xaxis(x)
    .add_yaxis('订单量',y)
    .set_global_opts(title_opts=opts.TitleOpts(title='每小时订单量走势'))
    .set_series_opts(
        label_opts=opts.LabelOpts(is_show=False),
        markpoint_opts=opts.MarkPointOpts(
            data=[
                opts.MarkPointItem(type_='max',name='峰值'),
                opts.MarkPointItem(name='第二峰值',coord=[x[15],y[15]],value=y[15]),
                opts.MarkPointItem(name='第三峰值',coord=[x[10],y[10]],value=y[10]),
            ]
        )
    )
)

In [ ]:
c.render('./OrderFromTmall/可视化图/每小时订单量走势图.html')

# **part 2**
- 双十一淘宝美妆数据.csv 这个数据集是美妆店铺的双十一销售数据，可以挖掘的纬度有日期、店铺，指标则有销售量、销售额、评论数等。

In [ ]:
data2 = pd.read_csv('./OrderFromTmall/data/双十一淘宝美妆数据.csv')
data2.head()

In [ ]:
data2.info()

In [ ]:
data2[data2.duplicated()].count()

In [ ]:
data2.drop_duplicates(inplace=True)             # 删除重复值
data2.reset_index(drop=True, inplace=True)      # 重建索引
data2.isnull().sum()                          # 查看空值

In [ ]:
data2.fillna(0,inplace=True)        # 填充空值
data2['update_time'] = pd.to_datetime(data2['update_time']).apply(lambda x:x.strftime('%Y-%m-%d'))      # 日期格式化，方便统计

In [ ]:
data2[data2['sale_count']>0].sort_values(by=['sale_count']).head()

In [ ]:
data2['sale_amount'] = data2['sale_count'] * data2['price']
data2[data2['sale_amount']>0].sort_values(by=['sale_count'])

## **2、数据分析与可视化**

### **2.1 每日整体销售量走势**

In [ ]:
result = data2.groupby('update_time').agg({'sale_count':'sum'}).to_dict()['sale_count']

c = (
    Line()
    .add_xaxis(list(result.keys()))
    .add_yaxis('销售量',list(result.values()))
    .set_series_opts(
        areastyle_opts=opts.AreaStyleOpts(opacity=0.5),
        label_opts=opts.LabelOpts(is_show=False),
        markpoint_opts=opts.MarkPointOpts(
            data=[
                opts.MarkPointItem(type_='max',name='最大值'),
                opts.MarkPointItem(type_='min',name='最小值'),
                opts.MarkPointItem(type_='average',name='平均值')
            ]
        )
    )
    .set_global_opts(title_opts=opts.TitleOpts(title='每日整体销售量走势'))
)

In [ ]:
c.render('./OrderFromTmall/可视化图/每日整体销售量走势图.html')

### **2.2 谁家的化妆品卖的最好**

In [ ]:
dts = list(data2['update_time'].unique())
dts.reverse()
dts

In [ ]:
t1 = Timeline()
t1.add_schema(
    is_loop_play=False,
    play_interval=500
)

for dt in dts:
    item = data2[data2['update_time']<=dt].groupby('店名').agg(sale_count_sum=('sale_count', 'sum'),sale_amount_sum=('sale_amount', 'sum')).sort_values(by='sale_count_sum',ascending=False).head(10).sort_values(by='sale_count_sum').to_dict()
    bar=(
        Bar()
        .add_xaxis([*item['sale_count_sum'].keys()])
        .add_yaxis('销售量',[round(val/10000,2) for val in item['sale_count_sum'].values()],label_opts=opts.LabelOpts(position='right',formatter='{@[1]/}万'))
        .add_yaxis('销售额',[round(val/10000/10000,2) for val in item['sale_amount_sum'].values()],label_opts=opts.LabelOpts(position='right',formatter='{@[1]/} 亿元'))
        .reversal_axis()
        .set_global_opts(
            title_opts=opts.TitleOpts(title = '累计销售量排行 TOP10')
        )
    )
    t1.add(bar,dt)

In [ ]:
t1.render('./OrderFromTmall/可视化图/累计销售量排行 TOP10图.html')

In [ ]:
item = data2.groupby('店名').agg({'sale_count':'sum'}).sort_values(by='sale_count',ascending=False).head(10).to_dict()['sale_count']
item = {k: round(v/10000,2) for k,v in item.items()}
c = (
    Pie()
    .add('销量',[*item.items()])
    .set_global_opts(
        title_opts=opts.TitleOpts(title='总销售量占比')
    )
    .set_series_opts(
        label_opts=opts.LabelOpts(formatter='{b}:{c} 万({d}%)') )
)

In [ ]:
c.render('./OrderFromTmall/可视化图/总销售量占比图.html')

### **2.4 谁家的化妆品最贵**

In [ ]:
item = data2.groupby('店名').agg({'price':'mean'}).sort_values(by='price',ascending=False).head(20).sort_values(by='price').to_dict()
c = (
    Bar()
    .add_xaxis([*item['price'].keys()])
    .add_yaxis('销售量',[round(v,2) for v in item['price'].values()],label_opts=opts.LabelOpts(position='right'))
    .reversal_axis()
    .set_global_opts(
        title_opts=opts.TitleOpts(title = '平均价格排行 TOP20')
    )
)

In [ ]:
c.render('./OrderFromTmall/可视化图/平均价格排行 TOP20图.html')

# **part3**
- 日化.xlsx 这个数据集是美妆类商品的订单数据，从数量来看，应该是批发类的订单。包含两个 sheet 页（订单表和商品表），可以挖掘的纬度有日期、地区、商品，指标则有销售量、销售额、增长率等。

## **1、数据理解与处理**

In [3]:
fact_order = pd.read_excel(r'./OrderFromTmall/data/日化.xlsx',sheet_name='销售订单表')
dim_product = pd.read_excel(r'./OrderFromTmall/data/日化.xlsx',sheet_name='商品信息表')

### **1.1 商品表数据清洗**

In [4]:
dim_product.head(10)

,商品编号,商品名称,商品小类,商品大类,销售单价
0,X001,商品1,面膜,护肤品,121
1,X002,商品2,面膜,护肤品,141
2,X003,商品3,面膜,护肤品,168
3,X004,商品4,面膜,护肤品,211
4,X005,商品5,面膜,护肤品,185
5,X006,商品6,面膜,护肤品,159
6,X007,商品7,面膜,护肤品,213
7,X008,商品8,面膜,护肤品,119
8,X009,商品9,面膜,护肤品,92
9,X010,商品10,面膜,护肤品,58


In [5]:
dim_product.describe()

,销售单价
count,122.000000
mean,156.155738
std,58.454619
min,56.000000
25%,102.250000
50%,158.000000
75%,210.750000
max,253.000000


In [6]:
dim_product[dim_product.duplicated()].count()

商品编号    0
商品名称    0
商品小类    0
商品大类    0
销售单价    0
dtype: int64

In [7]:
dim_product.isnull().sum()

商品编号    0
商品名称    0
商品小类    0
商品大类    0
销售单价    0
dtype: int64

In [8]:
fact_order.head()

,订单编码,订单日期,客户编码,所在区域,所在省份,所在地市,商品编号,订购数量,订购单价,金额
0,D31313,2019-05-16 00:00:00,S22796,东区,浙江省,台州市,X091,892,214,190888.0
1,D21329,2019-05-14 00:00:00,S11460,东区,安徽省,宿州市,X005,276,185,51060.0
2,D22372,2019-08-26 00:00:00,S11101,北区,山西省,忻州市,X078,1450,116,168200.0
3,D31078,2019-04-08 00:00:00,S10902,北区,吉林省,延边朝鲜族自治州,X025,1834,102,187068.0
4,D32470,2019-04-11 00:00:00,S18696,北区,北京市,北京市,X010,887,58,51446.0


In [12]:
fact_order.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31452 entries, 0 to 31451
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   订单编码    31452 non-null  object 
 1   订单日期    31452 non-null  object 
 2   客户编码    31452 non-null  object 
 3   所在区域    31450 non-null  object 
 4   所在省份    31450 non-null  object 
 5   所在地市    31452 non-null  object 
 6   商品编号    31451 non-null  object 
 7   订购数量    31450 non-null  object 
 8   订购单价    31448 non-null  object 
 9   金额      31448 non-null  float64
dtypes: float64(1), object(9)
memory usage: 2.4+ MB


In [10]:
fact_order[fact_order.duplicated()].count()

订单编码    6
订单日期    6
客户编码    6
所在区域    6
所在省份    6
所在地市    6
商品编号    6
订购数量    6
订购单价    6
金额      6
dtype: int64

In [13]:
fact_order.drop_duplicates(inplace=True)
fact_order.reset_index(drop=True, inplace=True)
fact_order.isnull().sum()

订单编码    0
订单日期    0
客户编码    0
所在区域    2
所在省份    2
所在地市    0
商品编号    1
订购数量    2
订购单价    4
金额      4
dtype: int64

In [14]:
fact_order.bfill(inplace=True)
fact_order.ffill(inplace=True)
fact_order.isnull().sum()

订单编码    0
订单日期    0
客户编码    0
所在区域    0
所在省份    0
所在地市    0
商品编号    0
订购数量    0
订购单价    0
金额      0
dtype: int64

In [17]:
fact_order['订单日期'] = fact_order['订单日期'].apply(lambda x:pd.to_datetime(x,format='%Y#%m#%d') if isinstance(x,str) else x)
fact_order[fact_order['订单日期'] > '2020-01-01']

,订单编码,订单日期,客户编码,所在区域,所在省份,所在地市,商品编号,订购数量,订购单价,金额
20797,D26533,2050-06-09,S21396,北区,河北省,石家庄市,X022,759,158,119922.0


In [18]:
fact_order = fact_order[fact_order['订单日期']<'2020-01-01']
fact_order['订单日期'].max(),fact_order['订单日期'].min()

(Timestamp('2019-09-30 00:00:00'), Timestamp('2019-01-01 00:00:00'))

In [20]:
fact_order['订购数量'] = fact_order['订购数量'].apply(lambda x:x.strip('个') if isinstance(x,str) else x).astype('int')
fact_order['订购单价'] = fact_order['订购单价'].apply(lambda x:x.strip('元') if isinstance(x,str) else x ).astype('float')
fact_order['金额'] = fact_order['金额'].astype('float')

In [21]:
fact_order.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31445 entries, 0 to 31445
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   订单编码    31445 non-null  object        
 1   订单日期    31445 non-null  datetime64[ns]
 2   客户编码    31445 non-null  object        
 3   所在区域    31445 non-null  object        
 4   所在省份    31445 non-null  object        
 5   所在地市    31445 non-null  object        
 6   商品编号    31445 non-null  object        
 7   订购数量    31445 non-null  int64         
 8   订购单价    31445 non-null  float64       
 9   金额      31445 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(6)
memory usage: 2.6+ MB


In [25]:
fact_order['所在省份'] = fact_order['所在省份'].str.replace('自治区|维吾尔|回族|壮族|省|市', '',regex=True)
fact_order['所在省份'].unique()

array(['浙江', '安徽', '山西', '吉林', '北京', '云南', '广东', '广西', '内蒙古', '新疆', '湖北',
       '江苏', '甘肃', '四川', '河南', '福建', '陕西', '辽宁', '山东', '江西', '重庆', '河北',
       '湖南', '上海', '贵州', '天津', '海南', '宁夏', '黑龙江'], dtype=object)

In [26]:
fact_order['客户编码'] = fact_order['客户编码'].str.replace('编码','',regex=True)

## **2、数据分析与可视化**

### **2.1 每月订购情况**

In [30]:
fact_order['订单月份'] = fact_order['订单日期'].apply(lambda x:x.month)
item = fact_order.groupby('订单月份').agg({'订购数量':'sum','金额':'sum'}).to_dict()
x = [f'{key} 月' for key in item['订购数量'].keys()]
y1 = [round(val/10000,2) for val in item['订购数量'].values()]
y2 = [round(val/10000/10000,2) for val in item['金额'].values()]
c = (
    Bar()
    .add_xaxis(x)
    .add_yaxis('订购数量(万元)',y1)
    .add_yaxis('金额(亿元)',y2)
    .set_global_opts(
        title_opts=opts.TitleOpts(title='每月订购情况'),
        legend_opts=opts.LegendOpts(
            selected_map = {'订购数量(万元)': True, '金额(亿元)': False}
        )
    )
    .set_series_opts(
        label_opts=opts.LabelOpts(is_show=False)
    )
)

In [31]:
c.render('./OrderFromTmall/可视化图/每月订购情况图.html')

'D:\\python\\PythonProject\\OrderFromTmall\\可视化图\\每月订购情况图.html'

### **2.2 哪里的人最爱美**

In [34]:
item = fact_order.groupby('所在地市').agg({'订购数量':'sum'}).sort_values(by='订购数量',ascending=False).head(20).sort_values(by='订购数量').to_dict()['订购数量']

c = (
    Bar()
    .add_xaxis([*item.keys()])
    .add_yaxis('订购量',[round(val/10000,2) for val in item.values()],label_opts=opts.LabelOpts(position='right',formatter='{@[1]/} 万'))
    .reversal_axis()
    .set_global_opts(
        title_opts=opts.TitleOpts(title='订购数量排行 TOP20')
    )
)

In [35]:
c.render('./OrderFromTmall/可视化图/订购数量排行 TOP20.html')

'D:\\python\\PythonProject\\OrderFromTmall\\可视化图\\订购数量排行 TOP20.html'

### **2.3 什么类型的美妆需求量最大**

In [37]:
order = pd.merge(fact_order,dim_product,on='商品编号',how='inner')
order

,订单编码,订单日期,客户编码,所在区域,所在省份,所在地市,商品编号,订购数量,订购单价,金额,订单月份,商品名称,商品小类,商品大类,销售单价
0,D31313,2019-05-16,S22796,东区,浙江,台州市,X091,892,214.0,190888.0,5,商品91,粉底,彩妆,214
1,D21329,2019-05-14,S11460,东区,安徽,宿州市,X005,276,185.0,51060.0,5,商品5,面膜,护肤品,185
2,D22372,2019-08-26,S11101,北区,山西,忻州市,X078,1450,116.0,168200.0,8,商品78,口红,彩妆,116
3,D31078,2019-04-08,S10902,北区,吉林,延边朝鲜族自治州,X025,1834,102.0,187068.0,4,商品25,眼霜,护肤品,102
4,D32470,2019-04-11,S18696,北区,北京,北京市,X010,887,58.0,51446.0,4,商品10,面膜,护肤品,58
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31439,D26211,2019-03-03,S14926,北区,辽宁,大连市,X014,1166,86.0,100276.0,3,商品14,面膜,护肤品,86
31440,D23434,2019-09-27,S18445,北区,河北,沧州市,X011,976,226.0,220576.0,9,商品11,面膜,护肤品,226
31441,D27724,2019-08-06,S15803,西区,贵州,黔东南州,X038,665,102.0,67830.0,8,商品38,面霜,护肤品,102
31442,D38486,2019-09-15,S12592,东区,浙江,杭州市,X019,665,102.0,67830.0,9,商品19,眼霜,护肤品,210


In [39]:
order.groupby(['商品大类','商品小类']).agg({'订购数量':'sum'}).sort_values(by=['商品大类','订购数量'],ascending=[True,False])

订购数量
商品大类 商品小类         
彩妆   口红    2013024
     粉底    1188621
     睫毛膏    587399
     眼影     296599
     蜜粉      45534
护肤品  面膜    5451914
     面霜    4566905
     爽肤水   3523687
     眼霜    3350743
     隔离霜   2488124
     防晒霜   2388610
     洁面乳   1928020

### **2.4 哪些省份的美妆需求量最大**

In [41]:
item = fact_order.groupby('所在地市').agg({'订购数量':'sum'}).to_dict()['订购数量']

c = (
    Map()
    .add('订购数量',[*item.items()],'china',is_map_symbol_show=False)
    .set_series_opts(label_opts= opts.LabelOpts(is_show=True))
    .set_global_opts(
        title_opts=opts.TitleOpts(title='省份分布'),
        visualmap_opts=opts.VisualMapOpts(max_=1000000)
    )
)

In [42]:
c.render('./OrderFromTmall/可视化图/省份分布图.html')

'D:\\python\\PythonProject\\OrderFromTmall\\可视化图\\省份分布图.html'

### **2.5 通过 RFM 模型挖掘客户价值**
- RFM 模型是衡量客户价值和客户创利能力的重要工具和手段，其中由3个要素构成了数据分析最好的指标，分别是：

    - R-Recency（最近一次购买时间）
    - F-Frequency（消费频率）
    - M-Money（消费金额）
- 设定一个计算权重，比如 R-Recency 20% F-Frequency 30% M-Money 50% ，最后通过这个权重进行打分，量化客户价值，后续还可以基于分数进一步打标签，用来指导二次营销的策略。

In [44]:
data_rfm = fact_order.groupby('客户编码').agg({'订单日期':'max','订单编码':'count','金额':'sum'})
data_rfm.columns = ['最近一次购买时间','消费频率','消费金额']

In [48]:
data_rfm['R'] = data_rfm['最近一次购买时间'].rank(pct=True)
data_rfm['F'] = data_rfm['消费频率'].rank(pct=True)
data_rfm['M'] = data_rfm['消费金额'].rank(pct=True)
data_rfm.sort_values(by='R',ascending=False)

,最近一次购买时间,消费频率,消费金额,R,F,M
客户编码,,,,,,
S19485,2019-09-30,9,1463984.0,0.980166,0.061808,0.109779
S21996,2019-09-30,24,3099394.0,0.980166,0.428506,0.386531
S17166,2019-09-30,17,3627037.0,0.980166,0.261993,0.478782
S10469,2019-09-30,30,4198071.0,0.980166,0.570572,0.564576
S22287,2019-09-30,3,720165.0,0.980166,0.002306,0.016605
...,...,...,...,...,...,...
S16503,2019-04-07,14,1682893.0,0.004613,0.198801,0.147601
S20864,2019-03-14,8,1118752.0,0.003229,0.040129,0.047970
S17547,2019-03-14,10,1784531.0,0.003229,0.088100,0.164207


In [49]:
data_rfm['score'] = data_rfm['R'] * 20 + data_rfm['F'] * 30 + data_rfm['M'] * 50
data_rfm['score'] = data_rfm['score'].round(1)
data_rfm.sort_values(by='score',ascending=False)

,最近一次购买时间,消费频率,消费金额,R,F,M,score
客户编码,,,,,,,
S17476,2019-09-30,69,10325832.0,0.980166,0.986624,0.987085,98.6
S22326,2019-09-30,62,10074609.0,0.980166,0.973247,0.984317,98.0
S11581,2019-09-28,79,10333668.0,0.918358,0.996771,0.988007,97.7
S12848,2019-09-29,66,9673572.0,0.944649,0.980627,0.980627,97.3
S19095,2019-09-26,81,11031632.0,0.864852,0.999077,0.996310,97.1
...,...,...,...,...,...,...,...
S11176,2019-06-09,7,614134.0,0.035978,0.023524,0.010148,1.9
S18379,2019-07-05,4,400195.0,0.071033,0.004151,0.005535,1.8
编号S13676,2019-07-12,1,334197.0,0.087177,0.000923,0.000923,1.8


- 根据这个分数结果，我们可以对客户打上一些标签，比如大于 80 分的，标志为优质客户，在资源有限的情况下，可以优先服务好优质客户。